%md
# 01 · Capa bronze
Carga de los 8 CSV de Home Credit (volumen `workspace.credit_risk.raw`) a tablas Delta `bronze_*`, sin modificar valores. Es idempotente: si una tabla ya existe, se omite.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.credit_risk;
CREATE VOLUME IF NOT EXISTS workspace.credit_risk.raw;

In [0]:
vol = "/Volumes/workspace/credit_risk/raw"
for f in dbutils.fs.ls(vol):
    print(f.name, round(f.size / 1e6), "MB")

In [0]:
esperado = {
    "application_train": 307511, "application_test": 48744,
    "bureau": 1716428, "bureau_balance": 27299925,
    "previous_application": 1670214, "installments_payments": 13605401,
    "credit_card_balance": 3840312, "POS_CASH_balance": 10001358,
}

for nombre, n_esp in esperado.items():
    tabla = f"workspace.credit_risk.bronze_{nombre.lower()}"
    if spark.catalog.tableExists(tabla):
        print(f"{tabla}: ya existe, se omite")
        continue
    df = spark.read.csv(f"{vol}/{nombre}.csv", header=True, inferSchema=True)
    df.write.mode("overwrite").saveAsTable(tabla)
    n = spark.table(tabla).count()
    print(f"{tabla}: {n:,} filas -> {'OK' if n == n_esp else 'REVISAR'}")